In [1]:
from pathlib import Path
import pandas as pd
import shutil
import os

# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(r"C:\Documents\GeoSigLIP")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR = PROJECT_ROOT / "kaggle_dataset"
IMAGE_DIR = OUTPUT_DIR / "images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Output:", OUTPUT_DIR)

Project: C:\Documents\GeoSigLIP
Output: C:\Documents\GeoSigLIP\kaggle_dataset


In [2]:
train_df = pd.read_csv(
    PROCESSED_DIR / "train.csv"
)

val_df = pd.read_csv(
    PROCESSED_DIR / "validation.csv"
)

test_df = pd.read_csv(
    PROCESSED_DIR / "test.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 2450
Validation: 1050
Test: 1750


In [3]:
all_df = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test")
    ],
    ignore_index=True
)

print("Total images:", len(all_df))

Total images: 5250


In [9]:
from pathlib import Path
import pandas as pd
import shutil
from tqdm.auto import tqdm

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(r"C:\Documents\GeoSigLIP")

SOURCE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DCID"
    / "DCID-512-7"
)

OUTPUT_DIR = PROJECT_ROOT / "kaggle_dataset"
IMAGE_DIR = OUTPUT_DIR / "images"

print("Source:", SOURCE_ROOT)
print("Source exists:", SOURCE_ROOT.exists())

# Recreate output image directory
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# LOAD MANIFESTS
# ============================================================

train_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "train.csv"
)

val_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "validation.csv"
)

test_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "test.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Source: C:\Documents\GeoSigLIP\data\raw\DCID\DCID-512-7
Source exists: True
Train: 2450
Validation: 1050
Test: 1750


In [11]:
from pathlib import Path

SOURCE_ROOT = (
    Path(r"C:\Documents\GeoSigLIP")
    / "data"
    / "raw"
    / "DCID"
    / "DCID-512-7"
)

image_files = list(SOURCE_ROOT.rglob("*.jpg"))

print("Images found:", len(image_files))

image_index = {
    img.name: img
    for img in image_files
}

print("Indexed images:", len(image_index))

Images found: 35000
Indexed images: 28000


In [12]:
# ============================================================
# COPY SELECTED IMAGES
# ============================================================

all_dfs = [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]

copied = 0
missing = []

for split, df in all_dfs:

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Copying {split}"
    ):

        original_path = Path(row["image_path"])
        filename = original_path.name

        # Find actual image using filename
        source = image_index.get(filename)

        if source is None:
            missing.append(filename)
            continue

        label = row["label"]

        destination_dir = IMAGE_DIR / label
        destination_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        destination = destination_dir / filename

        if not destination.exists():
            shutil.copy2(
                source,
                destination
            )

        copied += 1

print("\n==============================")
print("COPY COMPLETE")
print("==============================")
print("Copied:", copied)
print("Missing:", len(missing))

if missing:
    print("\nFirst missing images:")
    for x in missing[:20]:
        print(x)

Copying train:   0%|          | 0/2450 [00:00<?, ?it/s]

Copying validation:   0%|          | 0/1050 [00:00<?, ?it/s]

Copying test:   0%|          | 0/1750 [00:00<?, ?it/s]


COPY COMPLETE
Copied: 5250
Missing: 0


In [13]:
def create_portable_manifest(df):
    
    records = []
    
    for _, row in df.iterrows():
        
        source = Path(row["image_path"])
        
        records.append({
            "image_path": f"images/{row['label']}/{source.name}",
            "label": row["label"]
        })
    
    return pd.DataFrame(records)


portable_train = create_portable_manifest(train_df)
portable_val = create_portable_manifest(val_df)
portable_test = create_portable_manifest(test_df)

In [14]:
portable_train.to_csv(
    OUTPUT_DIR / "train.csv",
    index=False
)

portable_val.to_csv(
    OUTPUT_DIR / "validation.csv",
    index=False
)

portable_test.to_csv(
    OUTPUT_DIR / "test.csv",
    index=False
)

print("Manifests saved.")

Manifests saved.


In [15]:
print("=" * 60)
print("KAGGLE DATASET")
print("=" * 60)

print("\nTrain:", len(portable_train))
print("Validation:", len(portable_val))
print("Test:", len(portable_test))

print("\nClasses:")
print(
    portable_train["label"]
    .value_counts()
    .sort_index()
)

print("\nOutput directory:")
print(OUTPUT_DIR)

KAGGLE DATASET

Train: 2450
Validation: 1050
Test: 1750

Classes:
label
Basalt             350
Granite            350
Gray siltstone     350
Light sandstone    350
Marble             350
Mudstone           350
Red sandstone      350
Name: count, dtype: int64

Output directory:
C:\Documents\GeoSigLIP\kaggle_dataset


In [16]:
print("\nDataset structure:")

for item in sorted(OUTPUT_DIR.iterdir()):
    print(item.name)

print("\nImage folders:")

for item in sorted(IMAGE_DIR.iterdir()):
    count = len(list(item.glob("*")))
    print(f"{item.name}: {count}")


Dataset structure:
images
test.csv
train.csv
validation.csv

Image folders:
Basalt: 720
Granite: 720
Gray siltstone: 714
Light sandstone: 723
Marble: 723
Mudstone: 730
Red sandstone: 723


In [17]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("*"):
    if path.is_file():
        print(path)

In [18]:
from pathlib import Path
import shutil

PROJECT = Path(r"C:\Documents\GeoSigLIP")

SOURCE = PROJECT / "data" / "raw" / "DCID" / "DCID-512-7"
OUTPUT = PROJECT / "kaggle_dataset" / "images"

print("SOURCE EXISTS:", SOURCE.exists())
print("SOURCE:", SOURCE)

# Find all JPGs
images = list(SOURCE.rglob("*.jpg"))

print("TOTAL SOURCE IMAGES:", len(images))

# Copy ALL DCID-7 images, preserving class folders
for img in images:
    relative = img.relative_to(SOURCE)

    # relative looks like:
    # train/1.Red sandstone/image.jpg
    # or test/1.Red sandstone/image.jpg

    # We only need the class name
    class_name = relative.parts[1]

    destination_dir = OUTPUT / class_name
    destination_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    destination = destination_dir / img.name

    if not destination.exists():
        shutil.copy2(img, destination)

print("\nCOPY COMPLETE")

SOURCE EXISTS: True
SOURCE: C:\Documents\GeoSigLIP\data\raw\DCID\DCID-512-7
TOTAL SOURCE IMAGES: 35000

COPY COMPLETE


In [19]:
for folder in sorted(OUTPUT.iterdir()):
    if folder.is_dir():
        count = len(list(folder.glob("*.jpg")))
        print(f"{folder.name}: {count}")

print(
    "\nTOTAL:",
    sum(
        len(list(folder.glob("*.jpg")))
        for folder in OUTPUT.iterdir()
        if folder.is_dir()
    )
)

1.Red sandstone: 4000
2.Light sandstone: 4000
3.Gray siltstone: 4000
4.Mudstone: 4000
5.Granite: 4000
6.Basalt: 4000
7.Marble: 4000
Basalt: 720
Granite: 720
Gray siltstone: 714
Light sandstone: 723
Marble: 723
Mudstone: 730
Red sandstone: 723

TOTAL: 33053


In [20]:
from pathlib import Path
import pandas as pd
import shutil

PROJECT_ROOT = Path(r"C:\Documents\GeoSigLIP")

SOURCE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DCID"
    / "DCID-512-7"
)

OUTPUT_DIR = PROJECT_ROOT / "kaggle_dataset"
IMAGE_DIR = OUTPUT_DIR / "images"

# Recreate output
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Load manifests
train_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "train.csv"
)
val_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "validation.csv"
)
test_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "test.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# ------------------------------------------------------------
# IMPORTANT:
# Build a UNIQUE source mapping using the COMPLETE relative path
# instead of filename alone.
# ------------------------------------------------------------

source_files = list(SOURCE_ROOT.rglob("*.jpg"))

source_map = {}

for image in source_files:
    relative = image.relative_to(SOURCE_ROOT)

    # Example:
    # train/1.Red sandstone/image.jpg
    # test/1.Red sandstone/image.jpg
    key = str(relative).replace("\\", "/")

    source_map[key] = image

print("Source images indexed:", len(source_map))


def copy_split(df, split_name):
    records = []

    for _, row in df.iterrows():

        original = Path(row["image_path"])

        # Find the corresponding original path.
        # The original path contains DCID-512-7/train/... or test/...
        normalized = str(original).replace("\\", "/")

        if "DCID-512-7/" in normalized:
            relative_part = normalized.split(
                "DCID-512-7/",
                1
            )[1]
        else:
            raise ValueError(
                f"Cannot determine source path: {original}"
            )

        source = source_map.get(relative_part)

        if source is None:
            raise FileNotFoundError(
                f"Source image not found: {relative_part}"
            )

        label = row["label"]

        # Preserve split + label, preventing train/test collisions.
        destination_dir = (
            IMAGE_DIR
            / split_name
            / label
        )

        destination_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        destination = (
            destination_dir
            / source.name
        )

        shutil.copy2(
            source,
            destination
        )

        records.append({
            "image_path": (
                f"images/{split_name}/{label}/{source.name}"
            ),
            "label": label
        })

    return pd.DataFrame(records)


portable_train = copy_split(
    train_df,
    "train"
)

portable_val = copy_split(
    val_df,
    "validation"
)

portable_test = copy_split(
    test_df,
    "test"
)

# Save manifests
portable_train.to_csv(
    OUTPUT_DIR / "train.csv",
    index=False
)

portable_val.to_csv(
    OUTPUT_DIR / "validation.csv",
    index=False
)

portable_test.to_csv(
    OUTPUT_DIR / "test.csv",
    index=False
)

print("\nDONE")
print("Train:", len(portable_train))
print("Validation:", len(portable_val))
print("Test:", len(portable_test))

Train: 2450
Validation: 1050
Test: 1750
Source images indexed: 35000

DONE
Train: 2450
Validation: 1050
Test: 1750


In [21]:
train_paths = set(portable_train["image_path"])
val_paths = set(portable_val["image_path"])
test_paths = set(portable_test["image_path"])

print("Train ∩ Validation:", len(train_paths & val_paths))
print("Train ∩ Test:", len(train_paths & test_paths))
print("Validation ∩ Test:", len(val_paths & test_paths))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [22]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Documents\GeoSigLIP")
DATASET_DIR = PROJECT_ROOT / "kaggle_dataset"

zip_base = PROJECT_ROOT / "kaggle_dataset"
zip_path = Path(
    shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=DATASET_DIR.parent,
        base_dir=DATASET_DIR.name
    )
)

print("Created:")
print(zip_path)

print(
    f"Size: {zip_path.stat().st_size / 1024**3:.2f} GB"
)

Created:
C:\Documents\GeoSigLIP\kaggle_dataset.zip
Size: 0.19 GB


In [23]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Documents\GeoSigLIP")
DATASET_DIR = PROJECT_ROOT / "kaggle_dataset"

print("Dataset exists:", DATASET_DIR.exists())

# Check CSVs
for file in ["train.csv", "validation.csv", "test.csv"]:
    path = DATASET_DIR / file
    print(
        f"{file}:",
        "EXISTS" if path.exists() else "MISSING"
    )

# Check image directories
IMAGE_DIR = DATASET_DIR / "images"

print("\nImage directory:", IMAGE_DIR.exists())

if IMAGE_DIR.exists():
    print("\nImage folders and counts:")

    total = 0

    for folder in sorted(IMAGE_DIR.iterdir()):
        if folder.is_dir():

            count = len(list(folder.rglob("*.jpg")))

            print(f"{folder.name}: {count}")

            total += count

    print("\nTOTAL IMAGES:", total)

# Check CSV row counts
print("\nCSV rows:")

for file in ["train.csv", "validation.csv", "test.csv"]:
    df = pd.read_csv(DATASET_DIR / file)
    print(f"{file}: {len(df)}")

Dataset exists: True
train.csv: EXISTS
validation.csv: EXISTS
test.csv: EXISTS

Image directory: True

Image folders and counts:
test: 1750
train: 2450
validation: 1050

TOTAL IMAGES: 5250

CSV rows:
train.csv: 2450
validation.csv: 1050
test.csv: 1750


In [24]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Documents\GeoSigLIP")
DATASET_DIR = PROJECT_ROOT / "kaggle_dataset"

print("Dataset exists:", DATASET_DIR.exists())

print("\nCSV files:")
for file in ["train.csv", "validation.csv", "test.csv"]:
    path = DATASET_DIR / file
    print(f"{file}: {'EXISTS' if path.exists() else 'MISSING'}")

IMAGE_DIR = DATASET_DIR / "images"

print("\nImage directory exists:", IMAGE_DIR.exists())

if IMAGE_DIR.exists():
    print("\nImage structure:")

    total_images = 0

    for split in ["train", "validation", "test"]:
        split_dir = IMAGE_DIR / split

        if split_dir.exists():
            split_total = 0

            for class_dir in sorted(split_dir.iterdir()):
                if class_dir.is_dir():
                    count = len(list(class_dir.glob("*.jpg")))
                    print(f"{split}/{class_dir.name}: {count}")
                    split_total += count

            print(f"{split} TOTAL: {split_total}")
            total_images += split_total

    print(f"\nTOTAL IMAGES: {total_images}")

print("\nCSV row counts:")

for file in ["train.csv", "validation.csv", "test.csv"]:
    df = pd.read_csv(DATASET_DIR / file)
    print(f"{file}: {len(df)}")

Dataset exists: True

CSV files:
train.csv: EXISTS
validation.csv: EXISTS
test.csv: EXISTS

Image directory exists: True

Image structure:
train/Basalt: 350
train/Granite: 350
train/Gray siltstone: 350
train/Light sandstone: 350
train/Marble: 350
train/Mudstone: 350
train/Red sandstone: 350
train TOTAL: 2450
validation/Basalt: 150
validation/Granite: 150
validation/Gray siltstone: 150
validation/Light sandstone: 150
validation/Marble: 150
validation/Mudstone: 150
validation/Red sandstone: 150
validation TOTAL: 1050
test/Basalt: 250
test/Granite: 250
test/Gray siltstone: 250
test/Light sandstone: 250
test/Marble: 250
test/Mudstone: 250
test/Red sandstone: 250
test TOTAL: 1750

TOTAL IMAGES: 5250

CSV row counts:
train.csv: 2450
validation.csv: 1050
test.csv: 1750
